# Label-free estimates of the transfer causes (v7-ct-2026-10-05)
**Before running:** Session options → Accelerator **None** (CPU is enough), **Internet On**; Input → attach the dataset
containing **`construction_transfer_code_v7.zip`**. Then **Save Version → Save & Run All** and download `results_v7.zip` from Output.

In [ ]:
# 1. Copy the code from the attached dataset
import glob, os, shutil, zipfile
EXPECTED = 'v7-ct-2026-10-05'
W = '/kaggle/working/ct'
for z in glob.glob('/kaggle/input/**/construction_transfer_code_v7.zip', recursive=True):
    zipfile.ZipFile(z).extractall('/kaggle/tmp_ct')
versions = glob.glob('/kaggle/input/**/CODE_VERSION.txt', recursive=True) + glob.glob('/kaggle/tmp_ct/**/CODE_VERSION.txt', recursive=True)
found = {v: open(v).read().strip() for v in versions}
print('code versions found:', found)
good = [v for v, ver in found.items() if ver == EXPECTED]
assert good, f'Attach the dataset with construction_transfer_code_v7.zip (version {EXPECTED}). Found: {found}'
shutil.copytree(os.path.dirname(good[0]), W, dirs_exist_ok=True)
os.chdir(W)
print('using code from:', os.path.dirname(good[0]))

In [ ]:
# 2. Install PyTorch Geometric and download the 9 datasets (retried; a dataset that keeps failing is skipped)
!pip install -q torch_geometric
import time
from run import load
AVAILABLE, FAILED = [], []
for name in ('cora', 'citeseer', 'pubmed', 'photo', 'computers', 'cs', 'wikics', 'roman', 'ratings'):
    for attempt in range(5):
        try:
            X, y, _ = load(name, 'data')
            print(name, X.shape, 'classes', int(y.max()) + 1)
            AVAILABLE.append(name)
            break
        except Exception as e:
            print(f'{name}: attempt {attempt + 1} failed ({type(e).__name__}: {e})')
            if name == 'wikics':
                shutil.rmtree('data/WikiCS', ignore_errors=True)
            time.sleep(30 * (attempt + 1))
    else:
        FAILED.append(name)
print('available:', AVAILABLE, '| SKIPPED:', FAILED or 'none')

In [ ]:
# 3. Unit tests (expect 60 passed) + smoke test on Cora, seed 0
!python -m pytest -q tests
!python labelfree.py --datasets cora --seeds 0 --out /tmp/lf_smoke.csv 2>&1 | grep -v -i warn | tail -2
import pandas as pd
_s = pd.read_csv('/tmp/lf_smoke.csv')
assert len(_s) > 20 and _s.h_pseudo.notna().all(), 'smoke test failed'
print('SMOKE TEST OK. Mean abs error vs true homophily:',
      {c: round((_s[c] - _s.h_true).abs().mean(), 3) for c in ['h_em', 'h_lab', 'h_rule', 'h_pseudo']})

In [ ]:
# 4. Run on two CPU processes, printing progress live
import subprocess
os.makedirs('results', exist_ok=True)
GROUPS = {0: ['cora', 'citeseer', 'photo', 'computers', 'wikics'], 1: ['pubmed', 'cs', 'roman', 'ratings']}
GROUPS = {g: [d for d in ds if d in AVAILABLE] for g, ds in GROUPS.items()}
procs, logs = {}, {}
for g, ds in GROUPS.items():
    if ds:
        logs[g] = f'results/lf{g}.log'
        procs[g] = subprocess.Popen(['python', 'labelfree.py', '--datasets', *ds, '--out', f'results/labelfree_{g}.csv'],
                                    stdout=open(logs[g], 'w'), stderr=subprocess.STDOUT)
seen = {g: 0 for g in procs}
while True:
    for g in procs:
        lines = open(logs[g]).read().splitlines()
        for line in lines[seen[g]:]:
            if 'Warning' not in line:
                print(f'[p{g}] {line}', flush=True)
        seen[g] = len(lines)
    if all(p.poll() is not None for p in procs.values()):
        break
    time.sleep(30)
print('failed:', [g for g, p in procs.items() if p.returncode != 0] or 'none')

In [ ]:
# 5. Results zip (download results_v7.zip from the Output tab; the analysis is run locally with the v4 results)
!cd /kaggle/working && zip -qr results_v7.zip ct/results && ls -lh results_v7.zip